# EcomGPT-7B B-natural retry lr=5e-5 (resume từ step 125)

Model: nguynthanhquang/ecomgpt-7b (mount sẵn, không tải).

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no GPU'
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9, 1), 'GB')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
MODEL = '/kaggle/input/ecomgpt-7b'
if not (Path(MODEL)/'pytorch_model.bin.index.json').exists():
    hits = list(Path('/kaggle/input').rglob('pytorch_model.bin.index.json'))
    MODEL = str(hits[0].parent) if hits else MODEL
DATA_SRC = None
for c in Path('/kaggle/input').rglob('units_A.parquet'):
    DATA_SRC = c.parent; break
assert DATA_SRC, 'không thấy tiki-matrix-data'
ROOT = Path('/kaggle/working/matrix_data')
if ROOT.exists(): shutil.rmtree(ROOT)
(ROOT/'processed_A').mkdir(parents=True); (ROOT/'processed_B').mkdir(parents=True)
for n in ('A','B'):
    shutil.copy2(DATA_SRC/f'units_{n}.parquet', ROOT/f'processed_{n}/units.parquet')
    shutil.copy2(DATA_SRC/f'eval_{n}.parquet', ROOT/f'eval_{n}.parquet')
    shutil.copy2(DATA_SRC/f'meta_{n}.json', ROOT/f'meta_{n}.json')
TOOLS = None
for c in Path('/kaggle/input').rglob('train_ecomgpt.py'):
    TOOLS = c.parent; break
assert TOOLS, 'không thấy tools'
CKPT = None
for c in Path('/kaggle/input').rglob('adapter_config.json'):
    CKPT = c.parent; break
STEP = 0
if CKPT:
    f = CKPT/'STEP.txt'
    if f.exists(): STEP = int(f.read_text().strip() or 0)
print('MODEL', MODEL, '| TOOLS', TOOLS, '| CKPT', CKPT, 'step', STEP)


In [ ]:
OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(TOOLS/'train_ecomgpt.py'),
       '--dataset', 'B', '--data-root', str(ROOT), '--out', str(OUT),
       '--model', MODEL, '--seed', '42', '--tokens', '4000000', '--steps', '500',
       '--configs', 'natural', '--micro-batch', '2', '--max-seq', '960',
       '--save-every', '25', '--lr', '5e-5']
if STEP > 0:
    cmd += ['--init-adapter', str(CKPT), '--start-step', str(STEP)]
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)
src = OUT/'hyp_natural_seed42.parquet'
if src.exists():
    src.rename(OUT/'hyp_ecomgpt_b_natural_lr5e5_seed42.parquet')
print('DONE')
